# Generate 20-sample Factoid Candidate Banks for DPO

This notebook samples 20 stochastic answers per question from the Qwen 2.5 0.5B single-answer SFT adapter. It writes separate DPO-ready candidate banks for the training and 160-question development splits.


In [ ]:
import json
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'Task-Structured Counterfactual Preference Mining']
project_root = next((path for path in candidates if (path / 'src').is_dir()), None)
assert project_root is not None, 'Run this notebook from the project or workspace directory.'

MODEL_REF = project_root / 'Artifacts/Factoid_SFT/models/full_resources_single_answer_qwen25_05b/adapter'
TRAIN_INPUT = project_root / 'data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/train_prepared.json'
DEV_INPUT = project_root / 'data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json'
TRAIN_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_qwen25_05b_train_s3_n20'
DEV_OUTPUT_ROOT = project_root / 'Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_qwen25_05b_dev_s3_n20'
MODEL_SLUG = 'full-resources-single-answer-qwen25-05b'
SAMPLES_PER_QUESTION = 20
SKIP_COMPLETE_SPLITS = True
SPLITS = {
    'train': {'input': TRAIN_INPUT, 'output_root': TRAIN_OUTPUT_ROOT},
    'dev': {'input': DEV_INPUT, 'output_root': DEV_OUTPUT_ROOT},
}

for path in (MODEL_REF / 'adapter_config.json', TRAIN_INPUT, DEV_INPUT):
    assert path.exists(), f'Missing required input: {path}'

print(f'Model: {MODEL_REF}')
for split_name, split_config in SPLITS.items():
    candidate_bank = split_config['output_root'] / MODEL_SLUG / 'candidate_bank.jsonl'
    print(f'{split_name.title()} questions: {split_config["input"]}')
    print(f'{split_name.title()} output: {candidate_bank}')


## Sampling configuration

Twenty independently seeded samples are generated per question using temperature 0.7 and top-p 0.9. The agnostic parser is retained for candidate metadata because it also recovers numbered or partially tagged factoid outputs.

In [ ]:
import subprocess


def completed_candidate_bank(split_config: dict) -> bool:
    candidate_bank = split_config['output_root'] / MODEL_SLUG / 'candidate_bank.jsonl'
    manifest_path = split_config['output_root'] / MODEL_SLUG / 'manifest.json'
    if not candidate_bank.exists() or not manifest_path.exists():
        return False
    expected_questions = len(json.loads(split_config['input'].read_text(encoding='utf-8')))
    expected_candidates = expected_questions * SAMPLES_PER_QUESTION
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        actual_candidates = sum(1 for line in candidate_bank.open(encoding='utf-8') if line.strip())
    except (OSError, json.JSONDecodeError):
        return False
    return (
        manifest.get('dataset', {}).get('question_count') == expected_questions
        and manifest.get('candidate_count') == expected_candidates
        and actual_candidates == expected_candidates
    )


def generate_split(split_name: str, split_config: dict):
    command = [
        'python', '-m', 'cse_dpo.generate_candidate_bank',
        '--eval-input', str(split_config['input']),
        '--model-ref', str(MODEL_REF),
        '--output-dir', str(split_config['output_root']),
        '--dataset-name', 'bioasq-factoid',
        '--question-types', 'factoid',
        '--prompt-file', 'prompts/factoid_single_answer_aligned.json',
        '--prompt', 'factoid-single-answer-aligned-v1',
        '--chat-template', 'qwen-2.5',
        '--prompt-format', 'chat',
        '--samples-per-question-total', str(SAMPLES_PER_QUESTION),
        '--max-resources', '0',
        '--max-resource-chars', '0',
        '--resource-selection', 'first',
        '--resource-granularity', 'document',
        '--max-seq-length', '9000',
        '--max-new-tokens', '64',
        '--temperature', '0.7',
        '--top-p', '0.9',
        '--batch-size', '1',
        '--seed', '3407',
        '--factoid-parser-mode', 'agnostic',
        '--local-files-only',
    ]
    print(f'Generating {split_name} bank:')
    print(' '.join(command))
    subprocess.run(command, cwd=project_root, check=True)


for split_name, split_config in SPLITS.items():
    if SKIP_COMPLETE_SPLITS and completed_candidate_bank(split_config):
        print(f'Skipping complete {split_name} bank.')
        continue
    generate_split(split_name, split_config)


## Verify the DPO artifacts

Expected candidate counts are 28,800 for train (1,440 x 20) and 3,200 for dev (160 x 20).


In [ ]:
import json

for split_name, split_config in SPLITS.items():
    candidate_bank = split_config['output_root'] / MODEL_SLUG / 'candidate_bank.jsonl'
    manifest_path = split_config['output_root'] / MODEL_SLUG / 'manifest.json'
    assert candidate_bank.exists(), f'Candidate bank was not created: {candidate_bank}'
    assert manifest_path.exists(), f'Manifest was not created: {manifest_path}'
    with manifest_path.open(encoding='utf-8') as handle:
        manifest = json.load(handle)
    print(f'[{split_name}] Questions: {manifest["dataset"]["question_count"]}')
    print(f'[{split_name}] Candidates: {manifest["candidate_count"]}')
    print(f'[{split_name}] Samples per question: {manifest["generation"]["samples_per_question_total"]}')
    print(f'[{split_name}] Candidate bank: {candidate_bank}')
